# 공통유틸
* 래퍼(wrapper) — 복잡하거나 반복되는 호출을 한 함수로 감싸 간단한 인터페이스만 노출하는 것. `sk_llm(prompt)` 가 내부의 `client.models.generate_content(...)` 를 감춥니다.
* 모듈화 — 이런 공통 함수를 한 파일에 모아 import 해서 재사용하는 것.   
(예: `common.py/llm_utils.py`)

# LLM 호출 래퍼 만들기
`ask_llm(...)`를 앞으로 사용할 LLM 호출 래퍼를 만들고, 예외처리, 지수 백오프 재시도를 붙인다.

### 0. 준비

In [ ]:
import os, time   # time : 재시도 사이에 잠깐 '대기'할 때 사용
from google.colab import userdata
for _n in ('GEMINI_API_KEY', 'GOOGLE_API_KEY'):
    try:
        os.environ['GEMINI_API_KEY'] = userdata.get(_n); break   # 키를 환경변수로
    except Exception:
        pass
from google import genai
from google.genai import errors   # errors.APIError : SDK가 호출 실패 시 던지는 오류 타입(아래서 잡는다)
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

### 1. wrapper 처리
매번 model=, contents= 를 쓰는 번거로움을 한 줄 호출로 감싼다(공통 래퍼).


In [ ]:
# model에 기본모델을 지정하여, 부를 땐 프롬프트만 넘기면 되게 한다.
def ask_llm(prompt, model='gemini-2.5-flash-lite'):
    resp = client.models.generate_content(model=model, contents=prompt)
    return resp.text   # 호출부가 .text 를 신경쓰지 않도록 답변 텍스트만 돌려준다.

print(ask_llm('한 문장으로 자기소개 해줘.'))

저는 당신의 질문에 답하고 필요한 정보를 제공하며, 다양한 방식으로 도움을 드릴 수 있는 인공지능입니다.


### 3. 예외처리 + 지수 백오프 재시도
`google-genai` 는 API 오류를 `google.genai.errors.APIError` 로 던집니다.   
-> 일시적인 실패(네트워크 끊김·서버 혼잡 등)는 **지수백오프 재시도**(1→2→4초로 다시 실행)

In [ ]:
def ask_llm_safe(prompt, model='gemini-2.5-flash', retries=3):
    for attempt in range(1, retries + 1):          # 1, 2, 3회차까지 시도
        try:
            resp = client.models.generate_content(model=model, contents=prompt)
            return resp.text                        # 성공하면 즉시 반환(반복 종료)
        except errors.APIError as e:                # API 호출이 실패하면(예: 429 과부하)
            if attempt == retries:                  # 마지막 시도까지 실패면 더 숨기지 말고 예외를 올린다
                raise
            wait = 2 ** (attempt - 1)               # 1 → 2 → 4초로 점점 길게 = '지수 백오프'
            #  (실패 직후 곧바로 다시 들이대지 않고 간격을 늘려, 서버가 회복할 시간을 준다)
            print(f'재시도 {attempt}/{retries} (code={getattr(e, "code", "?")}), {wait}s 대기')
            time.sleep(wait)                        # wait 초 동안 멈춘 뒤 다음 시도

print(ask_llm_safe('반품 정책을 한 문장으로 요약해줘.'))

제품은 일정 기간 내에 원래 상태로 반품 시 환불 또는 교환이 가능합니다.


### 4. 모듈화
새 작업(요약)도 직접 API를 부르지 않고 위 안전 래퍼를 재사용한다.   
 → 재시도·예외처리가 자동으로 따라온다(한 곳에 모아둔 효과).

In [ ]:
def summarize(text):
    return ask_llm_safe(f'다음을 한 문장으로 요약:\n{text}')

print(summarize('고객이 사이즈가 안 맞아 교환을 원하며, 색상도 바꾸고 싶어합니다.'))

재시도 1/3 (code=503), 1s 대기
고객은 사이즈와 색상 변경을 위해 교환을 원합니다.


### 5. 에러처리,재시도에 쓰이는 현업 도구

In [ ]:
# tenacity: 범용 재시도 라이브러리
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.genai import errors

@retry(wait=wait_exponential(multiplier=1, max=10),
       stop=stop_after_attempt(3),
       retry=retry_if_exception_type(errors.ServerError))
def ask(prompt):
    return client.models.generate_content(model='gemini-2.5-flash', contents=prompt).text

In [ ]:
# 로그 수집,분석
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logging.getLogger('llm').info('호출 성공')